# Transport, Welfare and Inequality

**Purpose.** Evaluate welfare translation, inequality mitigation and scale-adjusted infrastructure residuals.

**Inputs.** Transport, macroeconomic, inequality and boundary data.

**Outputs.** Welfare and inequality estimates, maps and figures.

Run the code cells from top to bottom after updating the local paths described in the repository README. The notebook has no stored outputs; figures and tables are generated when it is executed.

In [ ]:
# Cell | Transport, welfare and demographic data assembly

# ============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
from matplotlib.colorbar import ColorbarBase
import geopandas as gpd
import contextily as ctx
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 25
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['figure.dpi'] = 300

# Map countries to the five African subregions used for aggregation.
region_mapping = {
    'North Africa':    ['Algeria', 'Egypt', 'Libya', 'Mauritania', 'Morocco', 'Tunisia'],
    'West Africa':     ['Benin', 'Burkina Faso', 'Ivory Coast', 'Gambia', 'Ghana',
                        'Guinea', 'Guinea-Bissau', 'Liberia', 'Mali', 'Niger',
                        'Nigeria', 'Senegal', 'Sierra Leone', 'Togo'],
    'Central Africa':  ['Cameroon', 'Central African Republic', 'Chad', 'Republic of the Congo',
                        'Democratic Republic of the Congo', 'Equatorial Guinea', 'Gabon'],
    'East Africa':     ['Burundi', 'Djibouti', 'Eritrea', 'Ethiopia', 'Kenya',
                        'Rwanda', 'Somalia', 'South Sudan', 'Sudan',
                        'United Republic of Tanzania', 'Uganda'],
    'Southern Africa': ['Angola', 'Botswana', 'eSwatini', 'Lesotho', 'Madagascar', 'Malawi',
                        'Mozambique', 'Namibia', 'South Africa', 'Zambia', 'Zimbabwe'],
}
country_to_region = {c: r for r, cs in region_mapping.items() for c in cs}
all_mapped_countries = list(country_to_region.keys())

region_palette = {
    'Central Africa':  '#E07B39',
    'East Africa':     '#3A9E6F',
    'North Africa':    '#4472C4',
    'Southern Africa': '#9B59B6',
    'West Africa':     '#C0392B',
}

df_transport = pd.read_excel(
    r'E:\Africa_Infrastructure_Data\transport_statistics_2015_2025.xlsx'
)
df_transport = df_transport[(df_transport['year'] >= 2015) & (df_transport['year'] <= 2024)].copy()
df_transport['region'] = df_transport['country'].map(country_to_region)
df_transport = df_transport[df_transport['country'].isin(all_mapped_countries)].copy()

road_type_cols = ['road_motorway_km', 'road_trunk_km', 'road_primary_km',
                  'road_secondary_km', 'road_tertiary_km']
df_transport['total_road_km'] = df_transport[road_type_cols].sum(axis=1)

df_econ_raw = pd.read_csv(r'E:\Africa_Infrastructure_Data\afdb\ObservationData_fhkvkjf_constant.csv')
afdb_map = {
    "Congo, Dem. Rep.": "Democratic Republic of the Congo",
    "Congo, Rep.":      "Republic of the Congo",
    "Tanzania":         "United Republic of Tanzania",
    "Eswatini":         "eSwatini",
    "Cote d'Ivoire":    "Ivory Coast",
}
df_econ_raw['country'] = df_econ_raw['country'].replace(afdb_map)
df_econ_raw = df_econ_raw[(df_econ_raw['Date'] >= 2015) & (df_econ_raw['Date'] <= 2024)]

def get_indicator(indicator_name, value_col='Value'):
    df = df_econ_raw[df_econ_raw['indicator'] == indicator_name][['country', 'Date', value_col]].copy()
    return df.rename(columns={'Date': 'year', value_col: indicator_name})

df_pop = get_indicator('Population, total')

# Read the spatial boundary layer.
gdf_shp = gpd.read_file(r'E:\Africa_Infrastructure_Data\boundaries_project_49\Country.shp')
gdf_shp['Land_Area_km2'] = gdf_shp.geometry.area / 1e6
shp_map = {"Côte d'Ivoire": 'Ivory Coast'}
gdf_shp['SOVEREIGNT'] = gdf_shp['SOVEREIGNT'].replace(shp_map)
df_area = gdf_shp[['SOVEREIGNT', 'Land_Area_km2']].rename(columns={'SOVEREIGNT': 'country'})

# Merge records using the shared country-year keys.
df = df_transport.merge(df_pop.rename(columns={'Population, total': 'Population'}),
                        on=['country', 'year'], how='left')
df = df.merge(df_area, on='country', how='left')

transport_cols   = ['total_road_km', 'motorway_trunk_paved_km', 'railway_length_km',
                    'railway_station_area_km2', 'port_area_km2', 'airport_area_km2']
transport_labels = ['Road', 'Paved Highway', 'Railway',
                    'Railway Station', 'Port', 'Aeroway']



In [ ]:
# Cell | Population-growth diagnostic
(df.groupby('year')['Population'].sum()[2024]-df.groupby('year')['Population'].sum()[2015])/df.groupby('year')['Population'].sum()[2015]

In [ ]:
# Cell | Scale-adjusted infrastructure residual maps

# Read the spatial boundary layer.
gdf = gpd.read_file(r'E:\Africa_Infrastructure_Data\boundaries_project_49\Country_Subregion.shp')
gdf['cx'] = gdf.geometry.centroid.x
gdf['cy'] = gdf.geometry.centroid.y
xmin_g, ymin_g, xmax_g, ymax_g = gdf.total_bounds
map_w = xmax_g - xmin_g
map_h = ymax_g - ymin_g

pad_x   = map_w * 0.04
pad_top = map_h * 0.03
pad_bot = map_h * (-0.06)
VX0, VX1 = xmin_g - pad_x, xmax_g + pad_x
VY0, VY1 = ymin_g - pad_bot, ymax_g + pad_top

def add_compass(ax, x=0.08, y=0.93, size=0.038):
    from matplotlib.patches import Polygon as P
    ax.add_patch(P(np.array([[x, y+size], [x-size*.4, y], [x+size*.4, y]]),
                   closed=True, fc='#222', ec='white', lw=.5,
                   transform=ax.transAxes, zorder=12))
    ax.add_patch(P(np.array([[x, y-size], [x-size*.4, y], [x+size*.4, y]]),
                   closed=True, fc='white', ec='#222', lw=.5,
                   transform=ax.transAxes, zorder=12))
    ax.text(x, y+size*1.3, 'N', transform=ax.transAxes,
            ha='center', va='center', fontsize=20, fontweight='bold',
            fontfamily='Times New Roman', color='#222', zorder=13)

def add_scalebar(ax, seg_km=1000, n_seg=2, map_width_m=None):
    bw = (seg_km * n_seg * 1000) / map_width_m if map_width_m else 0.30
    bx, by, bh = 0.04, 0.07, 0.012
    sw = bw / n_seg
    for i in range(n_seg):
        ax.add_patch(Rectangle((bx+i*sw, by), sw, bh,
                               fc='#111' if i%2==0 else '#eee',
                               ec='#333', lw=.6,
                               transform=ax.transAxes, zorder=11, clip_on=False))
    for i, km in enumerate([0, seg_km, seg_km*n_seg]):
        ax.text(bx+i*sw, by-.008,
                f'{km}' if km < seg_km*n_seg else f'{km} km',
                transform=ax.transAxes, ha='center', va='top', fontsize=20,
                fontfamily='Times New Roman', color='#333', zorder=12)

def compute_resid_country_mean(df_panel, y_col, pop_col='Population', area_col='Land_Area_km2'):
    """Estimate annual scaling residuals."""
    df_t = df_panel[['country', 'year', y_col, pop_col, area_col]].copy().dropna()
    df_t = df_t[(df_t[y_col] > 0) & (df_t[pop_col] > 0) & (df_t[area_col] > 0)]
    if len(df_t) < 20:
        return pd.Series(dtype=float)
        
    df_t['resid'] = np.nan
    
    # Process each study year in sequence.
    for year, group in df_t.groupby('year'):
        if len(group) > 3:
            X = np.column_stack([np.log(group[pop_col]), np.log(group[area_col])])
            y = np.log(group[y_col])
            model = LinearRegression().fit(X, y)
            # Calculate residual-based diagnostic quantities.
            df_t.loc[group.index, 'resid'] = y - model.predict(X)
            
    # Retain complete observations required for estimation.
    return df_t.dropna(subset=['resid']).groupby('country')['resid'].mean()

country_resid_all = {}
for y_col, label in zip(transport_cols, transport_labels):
    country_resid_all[y_col] = compute_resid_country_mean(df, y_col)
    n = len(country_resid_all[y_col].dropna())

cmap_div = plt.get_cmap('RdBu_r')
all_vals = [v for s in country_resid_all.values() for v in s.dropna()]
vmax = np.percentile(np.abs(all_vals), 95)
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

FIG_W = 24
data_aspect = (VX1 - VX0) / (VY1 - VY0)
col_w_frac  = (0.99 - 0.01 - 2*0.012) / 3
row_h_inch  = FIG_W * col_w_frac / data_aspect
legend_inch = 0.80
FIG_H = 2 * row_h_inch + legend_inch

fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')
gs = gridspec.GridSpec(2, 3, figure=fig,
                       left=0.01, right=0.99,
                       top=1.0 - legend_inch / FIG_H * 0.1,
                       bottom=legend_inch / FIG_H,
                       hspace=0.015, wspace=0.001)

axes = [[fig.add_subplot(gs[r, c]) for c in range(3)] for r in range(2)]

for idx, (y_col, label) in enumerate(zip(transport_cols, transport_labels)):
    row_i, col_i = divmod(idx, 3)
    ax = axes[row_i][col_i]
    ax.set_xlim(VX0, VX1)
    ax.set_ylim(VY0, VY1)

    country_resid = country_resid_all[y_col]

    # Continue gracefully if this estimation step fails.
    try:
        ctx.add_basemap(ax, crs=gdf.crs.to_string(),
                        source=ctx.providers.CartoDB.PositronNoLabels,
                        zoom='auto', zorder=1, attribution=False)
    except Exception:
        ax.set_facecolor('#F5F5F0')

    for _, row in gdf.iterrows():
        cname = row['国家']
        resid_val = country_resid.get(cname, np.nan) if cname in country_resid.index else np.nan
        if pd.isna(resid_val):
            face_color, alpha = '#D0D0D0', 0.6
        else:
            face_color, alpha = mcolors.to_hex(cmap_div(norm(resid_val))), 0.88
        gpd.GeoDataFrame([row], crs=gdf.crs).plot(
            ax=ax, color=face_color, edgecolor='none', alpha=alpha, zorder=2)

    gdf.boundary.plot(ax=ax, color='#444', linewidth=0.6, zorder=3)

    ax.axis('off')
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_color('#999'); sp.set_linewidth(0.8)
    ax.set_frame_on(True)

    ax.text(0.015, 0.985, chr(ord('a') + idx),
            transform=ax.transAxes, fontsize=35, fontweight='bold',
            va='top', fontfamily='Arial', color='#111', zorder=15)

    ax.text(0.985, 0.985, label,
            transform=ax.transAxes, ha='right', va='top',
            fontsize=27.5, fontweight='bold',
            fontfamily='Arial', color='#111', zorder=15)


    add_compass(ax)
    add_scalebar(ax, map_width_m=(VX1 - VX0))

    cax = ax.inset_axes([0.62, 0.04, 0.35, 0.018])
    cb = ColorbarBase(cax, cmap=cmap_div, norm=norm, orientation='horizontal')
    cb.ax.tick_params(labelsize=16)
    for sp in cb.ax.spines.values():
        sp.set_edgecolor('#aaa')

In [ ]:
# Cell | Annual transport-growth metrics

# Detect the available railway-station field across transport-data versions.
station_candidates = [c for c in df_transport.columns if 'station' in c.lower() or 'stop' in c.lower()]
if station_candidates:
    STATION_COL = station_candidates[0]
else:
    STATION_COL = 'railway_station_count'
    df_transport[STATION_COL] = 0

transport_6cols = ['total_road_km', 'motorway_trunk_paved_km', 'railway_length_km',
                   STATION_COL, 'port_area_km2', 'airport_area_km2']

transport_6labels = {
    'total_road_km':           'Road Length',
    'motorway_trunk_paved_km': 'Paved Highway Length',
    'railway_length_km':       'Railway Length',
    STATION_COL:               'Railway Station',
    'port_area_km2':           'Port Area',
    'airport_area_km2':        'Airport Area',
}


years_all = list(range(2015, 2025))  # 2015-2024

trans_base_country = df_transport[df_transport['year'] == 2015][
    ['country'] + transport_6cols].set_index('country')

trans_growth_by_year = {}
for yr in years_all:
    df_yr = df_transport[df_transport['year'] == yr][
        ['country'] + transport_6cols].set_index('country')

    growth = pd.DataFrame(index=trans_base_country.index)
    for col in transport_6cols:
        base_vals = trans_base_country[col]
        yr_vals   = df_yr[col]
        diff      = yr_vals - base_vals
        pct = diff / base_vals.replace(0, np.nan) * 100
        abs_growth = yr_vals
        growth[col + '_growth'] = pct.where(base_vals != 0, np.nan)

    trans_growth_by_year[yr] = growth.reset_index()

for yr in [2015, 2019, 2024]:
    for col in transport_6cols:
        vals = trans_growth_by_year[yr][col + '_growth'].dropna()



In [ ]:
# Cell | Transport-growth associations with welfare outcomes

from scipy import stats

_pop_df  = df_econ_raw[df_econ_raw['indicator'] == 'Population, total'][
               ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'Pop'})
_hfce_df = df_econ_raw[df_econ_raw['indicator'] == 'Household final consumption expenditure (current US$)'][
               ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'HFCE'})
_er_df   = df_econ_raw[df_econ_raw['indicator'] == 'Exchange rate (LCU per US$, period average)'][
               ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'ER'})
_cpi_df  = df_econ_raw[df_econ_raw['indicator'] == 'Inflation, consumer prices  index (2000 = 100)'][
               ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'CPI'})

_mc = (_pop_df.merge(_hfce_df, on=['country', 'year'], how='inner')
              .merge(_er_df,   on=['country', 'year'], how='left')
              .merge(_cpi_df,  on=['country', 'year'], how='left'))
_mc['HFCE_pc_curr'] = _mc['HFCE'] / _mc['Pop']
_base = _mc[_mc['year'] == 2015][['country', 'ER', 'CPI']].rename(
    columns={'ER': 'ER_2015', 'CPI': 'CPI_2015'})
_mc = _mc.merge(_base, on='country', how='left')
_mc['ER_ratio']  = (_mc['ER']  / _mc['ER_2015'] ).fillna(1)
_mc['CPI_ratio'] = (_mc['CPI_2015'] / _mc['CPI']).fillna(1)
_mc['cons_pc_2015'] = _mc['HFCE_pc_curr'] * _mc['ER_ratio'] * _mc['CPI_ratio']
df_cons2 = _mc[['country', 'year', 'cons_pc_2015']]

df_gdp2   = get_indicator('GDP (constant 2000 US$)')
df_gdppc2 = get_indicator('GDP per capita,(constant 2000 Prices, US $)')

df_econ_m = (df_gdp2
             .merge(df_gdppc2, on=['country', 'year'], how='outer')
             .merge(df_cons2,  on=['country', 'year'], how='outer'))
df_econ_m = df_econ_m[df_econ_m['country'].isin(all_mapped_countries)]

years_r = sorted(df_econ_m['year'].dropna().unique().astype(int))
df_cont_econ = df_econ_m.groupby('year').agg({
    'GDP (constant 2000 US$)':                     'sum',
    'GDP per capita,(constant 2000 Prices, US $)': 'mean',
    'cons_pc_2015':                                'mean',
}).reset_index().sort_values('year')

def growth_rate_r(series, base_year=2015):
    base = series[df_cont_econ['year'] == base_year].values[0]
    return ((series - base) / abs(base) * 100).values

gdp_gr   = growth_rate_r(df_cont_econ['GDP (constant 2000 US$)'])
gdppc_gr = growth_rate_r(df_cont_econ['GDP per capita,(constant 2000 Prices, US $)'])
cons_gr  = growth_rate_r(df_cont_econ['cons_pc_2015'])

df_trans_continent = df_transport.groupby('year')[transport_6cols].sum().reset_index()
trans_2015 = df_trans_continent[df_trans_continent['year'] == 2015][transport_6cols].iloc[0]

trans_growth_continent = []
for _, row in df_trans_continent.iterrows():
    yr = row['year']
    growth_row = {'year': yr}
    for col in transport_6cols:
        base_val = trans_2015[col]
        yr_val = row[col]
        if base_val != 0:
            growth_row[col + '_growth'] = (yr_val - base_val) / base_val * 100
        else:
            growth_row[col + '_growth'] = yr_val
    trans_growth_continent.append(growth_row)

df_trans_growth_cont = pd.DataFrame(trans_growth_continent)

for col in transport_6cols:
    col_growth = col + '_growth'
    vals = df_trans_growth_cont[col_growth]

df_cont_econ['gdp_growth'] = gdp_gr
df_cont_econ['gdppc_growth'] = gdppc_gr
df_cont_econ['cons_growth'] = cons_gr

scatter_data_list = []
for col in transport_6cols:
    col_growth = col + '_growth'
    df_temp = df_trans_growth_cont[['year', col_growth]].copy()
    df_temp = df_temp.merge(df_cont_econ[['year', 'gdp_growth', 'gdppc_growth', 'cons_growth']], 
                             on='year', how='inner')
    df_temp['infrastructure'] = transport_6labels[col]
    df_temp = df_temp.rename(columns={col_growth: 'trans_growth'})
    scatter_data_list.append(df_temp)

# Construct continental baseline-relative growth rates for welfare associations.
df_scatter_all = pd.concat(scatter_data_list, ignore_index=True)


all_trans_growth = df_scatter_all['trans_growth'].values
mean_trans = all_trans_growth.mean()
std_trans = all_trans_growth.std()
df_scatter_all['trans_growth_zscore'] = (df_scatter_all['trans_growth'] - mean_trans) / std_trans


transport_colors = {
    'Road Length':           '#FF6600',
    'Paved Highway Length':  '#00008B',
    'Railway Length':        '#00CED1',
    'Railway Station':       '#8B0000',
    'Port Area':             '#C71585',
    'Airport Area':          '#7B2FBE',
}

econ_colors2  = {'GDP': '#2166ac', 'GDP_PC': '#d6604d', 'HFCE_PC': '#f4a582'}
econ_markers2 = {'GDP': 'o',       'GDP_PC': 's',        'HFCE_PC': '^'}

FIG_W3 = 26
FIG_H3 = 6.0

fig3 = plt.figure(figsize=(FIG_W3, FIG_H3), facecolor='white')
gs = fig3.add_gridspec(1, 4, left=0.06, right=0.92, top=0.88, bottom=0.18, wspace=0.35)

axes3 = [fig3.add_subplot(gs[0, i]) for i in range(4)]

ax_a  = axes3[0]
ax_ar = ax_a.twinx()

ax_a.plot(years_r, gdp_gr,
          color=econ_colors2['GDP'], marker=econ_markers2['GDP'],
          lw=2, ms=6, label='GDP')
ax_a.set_ylabel('GDP Growth Rate (%)', fontsize=18, fontweight='bold',
                color=econ_colors2['GDP'], fontfamily='Arial')
ax_a.tick_params(axis='y', labelcolor=econ_colors2['GDP'], labelsize=16)

ax_ar.plot(years_r, gdppc_gr,
           color=econ_colors2['GDP_PC'], marker=econ_markers2['GDP_PC'],
           lw=2, ms=6, label='GDP_PC')
ax_ar.plot(years_r, cons_gr,
           color=econ_colors2['HFCE_PC'], marker=econ_markers2['HFCE_PC'],
           lw=2, ms=6, label='HFCE_PC')
ax_ar.set_ylabel('Per Capita Growth Rate (%)', fontsize=17, fontweight='bold',
                 color='#b2182b', fontfamily='Arial')
ax_ar.tick_params(axis='y', labelcolor='#b2182b', labelsize=16)

ax_a.axhline(0, color='#aaa', lw=0.8, ls='--')
ax_ar.axhline(0, color='#aaa', lw=0.8, ls='--')

ax_a.set_xticks(years_r)
ax_a.set_xticklabels(years_r, rotation=45, ha='right', fontsize=16, fontfamily='Arial')
ax_a.grid(axis='y', alpha=0.2, ls='--')
ax_a.spines['top'].set_visible(False)
ax_ar.spines['top'].set_visible(False)
for lbl in ax_a.get_yticklabels() + ax_ar.get_yticklabels():
    lbl.set_fontfamily('Arial')

lines1, labs1 = ax_a.get_legend_handles_labels()
lines2, labs2 = ax_ar.get_legend_handles_labels()
ax_a.legend(lines1 + lines2, labs1 + labs2,
            fontsize=15, loc='upper left', framealpha=0.85,
            labelspacing=0.2, handletextpad=0.4,
            prop={'family': 'Arial', 'size': 15})

ax_a.text(-0.18, 1.06, 'a', transform=ax_a.transAxes,
          fontsize=35, fontweight='bold', va='top',
          fontfamily='Arial', color='#111', ha='left')

def plot_scatter_with_overall_fit(ax, df_data, y_col, ylabel, label_letter, show_legend=False):
    """Helper function."""
    
    legend_handles = []
    legend_labels = []
    
    # Estimate the model for each transport indicator.
    for infra_type in transport_6labels.values():
        df_infra = df_data[df_data['infrastructure'] == infra_type]
        color = transport_colors[infra_type]
        
        scatter = ax.scatter(df_infra['trans_growth_zscore'], 
                             df_infra[y_col],
                             s=60, 
                             c=color, 
                             alpha=0.7,
                             edgecolors='white',
                             linewidths=0.5,
                             label=infra_type,
                             zorder=3)
        
        if show_legend:
            legend_handles.append(scatter)
            legend_labels.append(infra_type)
    
    x_all = df_data['trans_growth_zscore'].values
    y_all = df_data[y_col].values
    
    slope, intercept, r_value, p_value, std_err = stats.linregress(x_all, y_all)
    
    x_fit = np.linspace(-2.5, 2.5, 100)
    y_fit = slope * x_fit + intercept
    
    ax.plot(x_fit, y_fit, 
            color='#333333', 
            lw=2.5, 
            alpha=0.8,
            linestyle='-',
            zorder=4,
            label='Overall Fit')
    
    n = len(x_all)
    t_val = stats.t.ppf(0.975, n-2)  # 95% CI
    
    y_pred = slope * x_all + intercept
    residuals = y_all - y_pred
    s_res = np.sqrt(np.sum(residuals**2) / (n - 2))
    
    x_mean = np.mean(x_all)
    sxx = np.sum((x_all - x_mean)**2)
    
    se_fit = s_res * np.sqrt(1/n + (x_fit - x_mean)**2 / sxx)
    ci_upper = y_fit + t_val * se_fit
    ci_lower = y_fit - t_val * se_fit
    
    ax.fill_between(x_fit, ci_lower, ci_upper, 
                     color='#333333', 
                     alpha=0.15, 
                     zorder=2,
                     label='95% CI')
    
    ax.set_xlabel('Continental Transport Infrastructure\nStandardized Growth (Z-score)', 
                   fontsize=17, fontfamily='Arial')
    ax.set_ylabel(ylabel, fontsize=17, fontfamily='Arial')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-25, 25)
    ax.tick_params(labelsize=16)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', alpha=0.2, ls='--', zorder=1)
    
    ax.axvline(x=0, color='#999', linewidth=0.8, linestyle='--', alpha=0.5, zorder=1)
    
    ax.text(0.05, 0.95, f'r = {r_value:.3f}\np = {p_value:.3f}',
            transform=ax.transAxes, fontsize=15, va='top',
            fontfamily='Arial', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.text(-0.18, 1.06, label_letter, transform=ax.transAxes,
            fontsize=35, fontweight='bold', va='top',
            fontfamily='Arial', color='#111', ha='left')
    
    return legend_handles, legend_labels

plot_scatter_with_overall_fit(axes3[1], 
                               df_scatter_all,
                               'gdp_growth',
                               'GDP Growth Rate (%)',
                               'b',
                               show_legend=False)

plot_scatter_with_overall_fit(axes3[2],
                               df_scatter_all,
                               'gdppc_growth',
                               'GDP per Capita Growth Rate (%)',
                               'c',
                               show_legend=False)

handles, labels = plot_scatter_with_overall_fit(axes3[3],
                                                 df_scatter_all,
                                                 'cons_growth',
                                                 'HFCE per Capita Growth Rate (%)',
                                                 'd',
                                                 show_legend=True)

fig3.legend(handles, labels,
            loc='center right',
            bbox_to_anchor=(0.99, 0.5),
            fontsize=11,
            frameon=True,
            framealpha=0.95,
            edgecolor='#CCCCCC',
            fancybox=False,
            prop={'family': 'Arial', 'size': 11})

plt.show()


In [ ]:
# Cell | Transport-growth associations with wealth inequality

from scipy import stats

# Read the processed country-year wealth-inequality indicators.
df_ineq = pd.read_excel(r'E:\Africa_Infrastructure_Data\wealth_inequality_2015_2024.xlsx')
df_ineq = df_ineq[(df_ineq['year'] >= 2015) & (df_ineq['year'] <= 2024)]

df_pop_ineq = df_econ_raw[df_econ_raw['indicator'] == 'Population, total'][
    ['country', 'Date', 'Value']].rename(columns={'Date': 'year', 'Value': 'Population', 'country': 'country_std'})

# Merge records using the shared country-year keys.
df_ineq_pop = df_ineq.merge(df_pop_ineq, on=['country_std', 'year'], how='left')

def weighted_mean(group, value_col, weight_col='Population'):
    valid = group[[value_col, weight_col]].dropna()
    if len(valid) == 0 or valid[weight_col].sum() == 0:
        return np.nan
    return (valid[value_col] * valid[weight_col]).sum() / valid[weight_col].sum()

df_cont_ineq = df_ineq_pop.groupby('year').apply(
    lambda g: pd.Series({
        'top1_bottom50_ratio':  weighted_mean(g, 'top1_bottom50_ratio'),
        'top10_bottom50_ratio': weighted_mean(g, 'top10_bottom50_ratio'),
    })
).reset_index().sort_values('year')

def growth_rate_ineq(series, df_ref, base_year=2015):
    base = df_ref.loc[df_ref['year'] == base_year, series].values[0]
    return ((df_ref[series] - base) / abs(base) * 100).values

top1_gr  = growth_rate_ineq('top1_bottom50_ratio',  df_cont_ineq)
top10_gr = growth_rate_ineq('top10_bottom50_ratio', df_cont_ineq)
years_ineq = df_cont_ineq['year'].values.tolist()

for yr, v1, v10 in zip(years_ineq, top1_gr, top10_gr):
    print(f"  {yr}: top1/bottom50={v1:.2f}%,  top10/bottom50={v10:.2f}%")

df_ineq_years = pd.DataFrame({'year': years_ineq, 'top1_gr': top1_gr, 'top10_gr': top10_gr})

# Work on a copy to preserve the source table.
# Construct continental baseline-relative growth rates for welfare associations.
df_scatter_all_plot = df_scatter_all[df_scatter_all["year"] >= 2016].copy()
df_ineq_years_plot  = df_ineq_years[df_ineq_years["year"] >= 2016].copy()
# Aggregate inequality measures with population weights before model fitting.
df_scatter_ineq = df_scatter_all_plot.merge(df_ineq_years_plot, on="year", how="inner")

econ_fit_colors = {'GDP': '#333333', 'GDP_PC': '#d6604d', 'HFCE_PC': '#4393c3'}

FIG_W_NEW = 26
FIG_H_NEW = 6.0

fig_new = plt.figure(figsize=(FIG_W_NEW, FIG_H_NEW), facecolor='white')
gs_new = fig_new.add_gridspec(1, 4, left=0.06, right=0.98, top=0.88, bottom=0.18, wspace=0.25)
axes_new = [fig_new.add_subplot(gs_new[0, i]) for i in range(4)]

def poly2_fit_ci(x_all, y_all, x_fit):
    """Helper function."""
    n = len(x_all)
    X_mat = np.column_stack([np.ones(n), x_all, x_all**2])
    coeffs, _, _, _ = np.linalg.lstsq(X_mat, y_all, rcond=None)
    y_pred_all = X_mat @ coeffs
    s_res = np.sqrt(np.sum((y_all - y_pred_all)**2) / (n - 3))
    X_fit_mat = np.column_stack([np.ones(len(x_fit)), x_fit, x_fit**2])
    y_fit = X_fit_mat @ coeffs
    XtX_inv = np.linalg.inv(X_mat.T @ X_mat)
    se_fit = s_res * np.sqrt(np.einsum('ij,jk,ik->i', X_fit_mat, XtX_inv, X_fit_mat))
    t_val = stats.t.ppf(0.975, n - 3)
    return y_fit, y_fit - t_val * se_fit, y_fit + t_val * se_fit

def plot_fit_only(ax, df_data, y_col, fit_color, ci_color, label):
    x_all = df_data['trans_growth_zscore'].values
    y_all = df_data[y_col].values
    x_fit = np.linspace(-2.5, 2.5, 200)
    y_fit, ci_lo, ci_hi = poly2_fit_ci(x_all, y_all, x_fit)
    ax.fill_between(x_fit, ci_lo, ci_hi, color=ci_color, alpha=0.25, zorder=2)
    line = ax.plot(x_fit, y_fit, color=fit_color, lw=2.5, alpha=0.9,
                   linestyle='-', zorder=4, label=label)[0]
    return line

def plot_ineq_fit(ax, df_data, y_col, ylabel, label_letter, fit_color='#1a1a1a'):
    x_all = df_data['trans_growth_zscore'].values
    y_all = df_data[y_col].values
    x_fit = np.linspace(-2.5, 2.5, 200)
    y_fit, ci_lo, ci_hi = poly2_fit_ci(x_all, y_all, x_fit)
    ax.fill_between(x_fit, ci_lo, ci_hi, color=fit_color, alpha=0.15, zorder=2)
    ax.plot(x_fit, y_fit, color=fit_color, lw=2.5, alpha=0.85, zorder=4)

    ax.set_xlabel('Transport Infrastructure\nStandardized Growth (Z-score)',
                  fontsize=17, fontfamily='Arial')
    ax.set_ylabel(ylabel, fontsize=17, fontfamily='Arial')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-8, 8)
    ax.tick_params(labelsize=16)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', alpha=0.2, ls='--', zorder=1)
    ax.axvline(x=0, color='#999', linewidth=0.8, linestyle='--', alpha=0.5, zorder=1)
    ax.axhline(y=0, color='#999', linewidth=0.8, linestyle='--', alpha=0.5, zorder=1)
    ax.text(-0.18, 1.06, label_letter, transform=ax.transAxes,
            fontsize=35, fontweight='bold', va='top',
            fontfamily='Arial', color='#111', ha='left')

ax_a = axes_new[0]
line_gdp   = plot_fit_only(ax_a, df_scatter_all_plot, 'gdp_growth',
                           econ_fit_colors['GDP'],     econ_fit_colors['GDP'],     'GDP')
line_gdppc = plot_fit_only(ax_a, df_scatter_all_plot, 'gdppc_growth',
                           econ_fit_colors['GDP_PC'],  econ_fit_colors['GDP_PC'],  'GDP per Capita')
line_cons  = plot_fit_only(ax_a, df_scatter_all_plot, 'cons_growth',
                           econ_fit_colors['HFCE_PC'], econ_fit_colors['HFCE_PC'], 'HFCE per Capita')

ax_a.set_xlabel('Transport Infrastructure\nStandardized Growth (Z-score)',
                fontsize=19, fontfamily='Arial')
ax_a.set_ylabel('Growth Rate (%)', fontsize=19, fontfamily='Arial')
ax_a.set_xlim(-3, 3)
ax_a.set_ylim(-25, 25)
ax_a.tick_params(labelsize=16)
ax_a.spines['top'].set_visible(False)
ax_a.spines['right'].set_visible(False)
ax_a.grid(axis='both', alpha=0.2, ls='--', zorder=1)
ax_a.axvline(x=0, color='#999', linewidth=0.8, linestyle='--', alpha=0.5, zorder=1)
ax_a.legend([line_gdp, line_gdppc, line_cons],
            ['GDP', 'GDP_PC', 'HFCE_PC'],
            loc='upper left', fontsize=16, frameon=True, framealpha=0.95,
            bbox_to_anchor=(0.01, 1.03),
            handletextpad=0.5,
            labelspacing=0.2,
            borderpad=0.3,
            edgecolor='#CCCCCC', fancybox=False,
            prop={'family': 'Arial', 'size': 16})
ax_a.text(-0.18, 1.06, 'g', transform=ax_a.transAxes,
          fontsize=35, fontweight='bold', va='top', fontfamily='Arial', color='#111', ha='left')

plot_ineq_fit(axes_new[1], df_scatter_ineq, 'top1_gr',
              'Weighted Top1/Bottom50 Ratio\nGrowth Rate (%)', 'h', fit_color='#8B0000')

plot_ineq_fit(axes_new[2], df_scatter_ineq, 'top10_gr',
              'Weighted Top10/Bottom50 Ratio\nGrowth Rate (%)', 'i', fit_color='#00008B')

ax_d = axes_new[3]

legend_lines = []
legend_labs  = ['Road', 'Paved Highway', 'Railway', 'Railway Station', 'Port', 'Aeroway']

df_cont_trans_growth = df_trans_growth_cont[['year'] + [c + '_growth' for c in transport_6cols]].copy()
df_d_base = df_cont_trans_growth[df_cont_trans_growth["year"] >= 2016].merge(df_ineq_years_plot[['year', 'top10_gr']], on='year', how='inner')

for col in transport_6cols:
    label  = transport_6labels[col]
    color  = transport_colors[label]
    col_gr = col + '_growth'

    x_raw = df_d_base[col_gr].values.astype(float)
    y_raw = df_d_base['top10_gr'].values.astype(float)

    x_mean, x_std = x_raw.mean(), x_raw.std()
    x_z = (x_raw - x_mean) / x_std if x_std > 0 else x_raw - x_mean

    x_fit = np.linspace(x_z.min() - 0.3, x_z.max() + 0.3, 200)
    try:
        y_fit, ci_lo, ci_hi = poly2_fit_ci(x_z, y_raw, x_fit)
        ax_d.fill_between(x_fit, ci_lo, ci_hi, color=color, alpha=0.12, zorder=2)
        line, = ax_d.plot(x_fit, y_fit, color=color, lw=2.2, alpha=0.9, zorder=4)
        legend_lines.append(line)
    except Exception as e:
        pass

ax_d.set_xlabel('Per-Type Standardized\nGrowth Rate (Z-score)',
                fontsize=19, fontfamily='Arial')
ax_d.set_ylabel('Weighted Top10/Bottom50 Ratio\nGrowth Rate (%)', fontsize=19, fontfamily='Arial')
ax_d.set_xlim(-3, 3)
ax_d.set_ylim(-8, 4)
ax_d.tick_params(labelsize=16)
ax_d.spines['top'].set_visible(False)
ax_d.spines['right'].set_visible(False)
ax_d.grid(axis='both', alpha=0.2, ls='--', zorder=1)
ax_d.axvline(x=0, color='#999', linewidth=0.8, linestyle='--', alpha=0.5, zorder=1)
ax_d.axhline(y=0, color='#999', linewidth=0.8, linestyle='--', alpha=0.5, zorder=1)z
ax_d.legend(legend_lines, legend_labs,
            handletextpad=0.5,
            labelspacing=0.2,
            borderpad=0.3,
            loc='lower left', fontsize=16, frameon=True, framealpha=0.95,
            edgecolor='#CCCCCC', fancybox=False,
            prop={'family': 'Arial', 'size': 16})
ax_d.text(-0.12, 1.06, 'j', transform=ax_d.transAxes,
          fontsize=35, fontweight='bold', va='top',
          fontfamily='Arial', color='#111', ha='left')

plt.show()
